<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drexelwireless/ece512-courseware/blob/main/notebooks/lectures/week03-capacity-trunking.ipynb)

# Week 3 — Spectral Efficiency, Trunking, and Erlang B/C
**ECE 512 Wireless Communications** · companion notebook to the Week 3 lecture

This notebook follows the order of the slides:

1. Cellular spectral efficiency $\eta_S = \eta_B\,\eta_C\,\eta_T$
2. Trunking and the Erlang-B / Erlang-C formulas (GoS chart, channels needed)
3. Trunking efficiency: what sectoring does to a trunk
4. Discrete-time Markov chains and the stationary distribution $\boldsymbol{\pi} = \boldsymbol{\pi}\mathbf{P}$
5. The Poisson process and exponential inter-arrival times
6. A one-channel trunk (the POTS line) and the balance equations
7. Erlang B: the M/M/$c$/$c$ loss system, theory vs. simulation
8. Erlang C: the M/M/$c$/$\infty$ delay system, theory vs. simulation

Run the setup cell first. In Colab: *Runtime → Run all*.

In [ ]:
# --- Setup: installs the course package in Google Colab (safe to re-run) ---
COURSEWARE_URL = "git+https://github.com/drexelwireless/ece512-courseware"
import importlib.util
if importlib.util.find_spec("ece512") is None:
    get_ipython().run_line_magic("pip", f"install -q {COURSEWARE_URL}")

import math
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, FloatSlider, FloatLogSlider
from ece512 import trunking as tr
from ece512.cells import cluster_size

rng = np.random.default_rng(512)

## 1. Cellular spectral efficiency
For an FDMA system the lecture writes the spectral efficiency (Erlangs/m$^2$/Hz) as a product of three factors:
$$\eta_S = \eta_B\,\eta_C\,\eta_T .$$
One concrete way to split it: a total bandwidth $W$ is cut into channels of width $W_{\mathrm{ch}}$, a cluster of
$N$ cells shares them, so each cell of area $A_{\mathrm{cell}} = \tfrac{3\sqrt3}{2}R^2$ has $c = \lfloor W/(W_{\mathrm{ch}}N)\rfloor$
channels and can carry $A$ Erlangs at the target grade of service (Erlang B, Section 2). Then

| factor | meaning | here |
|---|---|---|
| $\eta_B$ bandwidth efficiency | channels per Hz (modulation/signaling) | $1/W_{\mathrm{ch}}$ |
| $\eta_C$ spatial efficiency | how densely a channel is reused | $1/(N A_{\mathrm{cell}})$ |
| $\eta_T$ trunking efficiency | offered traffic per channel | $A/c$ |

and $\eta_S \approx A/(W A_{\mathrm{cell}})$. Small $N$ improves $\eta_C$ *and* $\eta_T$ (bigger trunks), but the first-tier
SIR $\Lambda \approx (\sqrt{3N})^\beta/6$ (Week 2) limits how small $N$ may be. Gray bars below violate the SIR requirement.

In [ ]:
N_ALLOWED = sorted({cluster_size(i, j) for i in range(6) for j in range(6)} - {0})
N_ALLOWED = [n for n in N_ALLOWED if n <= 28]

def show_spectral_efficiency(W_MHz=12.5, Wch_kHz=30.0, R_km=1.0, beta=4.0, sir_min_dB=18.0, gos=0.02):
    area = tr.hex_cell_area(R_km * 1e3)
    rows = [tr.spectral_efficiency(W_MHz * 1e6, Wch_kHz * 1e3, n, area, gos) for n in N_ALLOWED]
    sir_dB = 10 * np.log10(tr.first_tier_sir(np.array(N_ALLOWED), beta))
    ok = sir_dB >= sir_min_dB
    etaS = np.array([r["eta_S"] for r in rows]) * 1e12          # Erl/km^2/MHz
    etaT = np.array([r["eta_T"] for r in rows])
    x = np.arange(len(N_ALLOWED))
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
    for ax, y, lab in [(axes[0], etaS, r"$\eta_S$ (Erlangs/km$^2$/MHz)"),
                       (axes[1], etaT, r"$\eta_T = A/c$ (Erlangs/channel)")]:
        ax.bar(x, y, color=np.where(ok, "C0", "lightgray"), edgecolor="white", linewidth=1)
        ax.set_xticks(x, [str(n) for n in N_ALLOWED]); ax.set_xlabel("cluster size N")
        ax.set_ylabel(lab); ax.grid(axis="y", alpha=0.3)
    if ok.any():
        k = int(np.argmax(ok))
        axes[0].annotate(f"best feasible: N={N_ALLOWED[k]}\n{etaS[k]:.3f} Erl/km²/MHz",
                         xy=(k, etaS[k]), xytext=(k + 2, etaS.max() * 0.8),
                         arrowprops=dict(arrowstyle="->"))
    axes[0].set_title(f"Spectral efficiency (GoS {100*gos:g}%, blue: SIR ≥ {sir_min_dB:g} dB)")
    axes[1].set_title("Trunking efficiency falls as N grows (smaller trunks)")
    plt.tight_layout(); plt.show()
    print(f"{'N':>3} {'SIR dB':>7} {'c':>5} {'A (Erl)':>8} {'eta_B (ch/MHz)':>15} {'eta_C (1/km^2)':>15} {'eta_T':>6}")
    for n, s, r in zip(N_ALLOWED, sir_dB, rows):
        print(f"{n:>3} {s:7.1f} {r['c']:>5} {r['A']:8.1f} {r['eta_B']*1e6:15.1f} {r['eta_C']*1e6:15.4f} {r['eta_T']:6.3f}")

interact(show_spectral_efficiency,
         W_MHz=FloatSlider(value=12.5, min=1.25, max=25, step=1.25, description="W (MHz)"),
         Wch_kHz=FloatSlider(value=30, min=10, max=200, step=5, description="W_ch (kHz)"),
         R_km=FloatSlider(value=1.0, min=0.2, max=5, step=0.1, description="R (km)"),
         beta=FloatSlider(value=4.0, min=2, max=5, step=0.1, description="β"),
         sir_min_dB=FloatSlider(value=18, min=5, max=25, step=0.5, description="SIR min (dB)"),
         gos=FloatSlider(value=0.02, min=0.005, max=0.1, step=0.005, description="GoS"));

Spatial efficiency is also about **cell area**: $\eta_C \propto 1/(N R^2)$, so halving the cell radius (cell splitting)
quadruples $\eta_S$ for the same spectrum. The static plot below compares the knobs on one log scale.

In [ ]:
R_km = np.logspace(-1, 1, 60)
fig, ax = plt.subplots(figsize=(7, 4))
for i, (n, wch) in enumerate([(7, 30e3), (4, 30e3), (7, 10e3)]):
    eta = [tr.spectral_efficiency(12.5e6, wch, n, tr.hex_cell_area(r * 1e3))["eta_S"] * 1e12 for r in R_km]
    ax.loglog(R_km, eta, color=f"C{i}", lw=2, label=f"N = {n}, $W_{{\\mathrm{{ch}}}}$ = {wch/1e3:g} kHz")
ax.set_xlabel("cell radius R (km)"); ax.set_ylabel(r"$\eta_S$ (Erlangs/km$^2$/MHz)")
ax.set_title(r"$\eta_S \propto 1/R^2$: cell splitting beats every other knob")
ax.grid(True, which="both", alpha=0.3); ax.legend(); plt.show()

## 2. Trunking and the Erlang formulas
With $U$ users, each requesting $\lambda_u$ calls/s with mean holding time $1/\mu$, the offered traffic is
$\rho = U\lambda_u/\mu$ Erlangs (in the derivation $\lambda = U\lambda_u$ is the arrival rate over **all** users, $\rho = \lambda/\mu$).
For a trunk of $c$ channels:

* **Blocked calls cleared** (M/M/$c$/$c$): $\displaystyle B(\rho,c) = \frac{\rho^c}{c!\sum_{k=0}^{c}\frac{\rho^k}{k!}}$
* **Blocked calls delayed** (M/M/$c$/$\infty$): $\displaystyle C(\rho,c) = \frac{\rho^c}{\rho^c + c!\left(1-\frac{\rho}{c}\right)\sum_{k=0}^{c-1}\frac{\rho^k}{k!}}$

The factorials overflow quickly, so `ece512.trunking` evaluates Erlang B with the stable recursion
$B(\rho,0)=1,\; B(\rho,k) = \dfrac{\rho\,B(\rho,k-1)}{k + \rho\,B(\rho,k-1)}$ and Erlang C as $C = \dfrac{cB}{c-\rho(1-B)}$.

In [ ]:
# Example: 2000 users, each 2 calls/hour of 3 minutes on average
U, calls_per_hr, H_min = 2000, 2, 3
rho_user = calls_per_hr / 60 * H_min
print(f"user traffic = {rho_user:.3f} Erl, offered traffic rho = {U * rho_user:.0f} Erl")
for c in [100, 110, 120]:
    print(f"  c = {c}: B = {tr.erlang_b(U * rho_user, c):.4f},  C = {tr.erlang_c(U * rho_user, c):.4f}")

print("\nWhy the recursion? Slide formula in float64 vs recursion:")
for rho, c in [(10, 12), (120, 150), (150, 200), (900, 1000)]:
    print(f"  rho={rho:>4}, c={c:>4}:  direct = {tr.erlang_b_direct(rho, c):.6g},  recursion = {tr.erlang_b(rho, c):.6g}")

The slide's table (from Rappaport) lists the traffic a trunk can carry at a given grade of service (GoS). We recover it by
inverting $B(A,c) = \text{GoS}$ numerically:

In [ ]:
slide = {2: [0.153, 0.105, 0.065, 0.046], 4: [0.869, 0.701, 0.535, 0.439], 5: [1.36, 1.13, 0.900, 0.762],
         10: [4.46, 3.96, 3.43, 3.09], 20: [12.0, 11.1, 10.1, 9.41], 24: [15.3, 14.2, 13.0, 12.2],
         40: [29.0, 27.4, 25.6, 24.4], 70: [56.1, 53.7, 51.0, 49.2], 100: [84.1, 80.9, 77.5, 75.2]}
gos_list = [0.01, 0.005, 0.002, 0.001]
print(f"{'C':>4} | " + " | ".join(f"GoS={g:<6}" for g in gos_list) + "   (computed / slide)")
for c, vals in slide.items():
    print(f"{c:>4} | " + " | ".join(f"{tr.offered_traffic_for_gos(c, g):6.3f}/{v:<6}" for g, v in zip(gos_list, vals)))

**Erlang B chart.** A Rappaport-style GoS chart: each gray curve is one trunk size $c$. Pick $c$ and a GoS to read off the
capacity; notice how much more traffic *per channel* a large trunk carries at the same GoS (diminishing returns).

In [ ]:
C_CHART = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 14, 16, 18, 20, 25, 30, 35, 40, 50, 60, 70, 80, 90, 100]
A_AX = np.logspace(-1, 2, 400)

def erlang_chart(ax, fn, c_sel, ylim=(1e-3, 0.1)):
    for c in C_CHART:
        ax.loglog(A_AX, fn(A_AX, c), color="0.7", lw=0.8)
        a_top = tr.offered_traffic_for_gos(c, ylim[1]) if fn is tr.erlang_b else None
        if a_top is not None and A_AX[0] < a_top < A_AX[-1] and c not in (9, 14, 18, 35, 70, 90):
            ax.text(a_top, ylim[1] * 1.07, str(c), fontsize=8, ha="center", va="bottom", color="0.35")
    name = "Erlang B" if fn is tr.erlang_b else "Erlang C"
    ax.loglog(A_AX, fn(A_AX, c_sel), color="C1", lw=2.2, label=f"{name}, c = {c_sel}")
    ax.set_ylim(*ylim); ax.set_xlim(A_AX[0], A_AX[-1])
    ax.set_xlabel("offered traffic A (Erlangs)"); ax.grid(True, which="both", alpha=0.25)

def show_erlang_b_chart(c=10, gos=0.02):
    fig, ax = plt.subplots(figsize=(8, 6))
    erlang_chart(ax, tr.erlang_b, c)
    a = tr.offered_traffic_for_gos(c, gos)
    ax.axhline(gos, color="C0", ls="--", lw=1, label=f"GoS = {gos:g}")
    ax.plot(a, gos, "o", color="C1", ms=9, mec="white", mew=2)
    ax.annotate(f"A = {a:.2f} Erl\n({a/c:.2f} Erl/channel)", xy=(a, gos), xytext=(12, -38),
                textcoords="offset points", fontsize=10)
    ax.set_ylabel("blocking probability B(A, c)")
    ax.set_title("Erlang B (blocked calls cleared); numbers on top = channels c", pad=22)
    ax.legend(loc="lower right"); plt.show()

interact(show_erlang_b_chart, c=IntSlider(value=10, min=1, max=100, step=1),
         gos=FloatLogSlider(value=0.02, base=10, min=-3, max=-1, step=0.05, description="GoS"));

**Erlang C chart** (blocked calls delayed) for the same trunk sizes: $C(\rho,c) \ge B(\rho,c)$ always — a call that would
have been blocked now waits, which keeps the channels busier.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
erlang_chart(ax, tr.erlang_c, 10, ylim=(1e-3, 1.0))
ax.loglog(A_AX, tr.erlang_b(A_AX, 10), color="C0", lw=1.8, ls="--", label="Erlang B, c = 10 (for comparison)")
ax.set_ylabel("probability of delay C(A, c)")
ax.set_title("Erlang C for c = 1 … 100 (gray); c = 10 highlighted")
ax.legend(loc="lower right"); plt.show()

**Channels needed.** The inverse question a planner asks: how many channels are needed to carry $A$ Erlangs at 2% blocking?
The dashed line $c = A$ shows the (unreachable) no-blocking ideal; the gap shrinks relative to $A$ as the trunk grows.

In [ ]:
A = np.linspace(0.1, 100, 400)
fig, ax = plt.subplots(figsize=(8, 4.5))
for i, g in enumerate([0.005, 0.02, 0.05]):
    ax.step(A, [tr.channels_for_gos(a, g) for a in A], where="post", color=f"C{i}", lw=1.8, label=f"GoS = {100*g:g}%")
ax.plot(A, A, "k--", lw=1, label="c = A")
ax.set_xlabel("offered traffic A (Erlangs)"); ax.set_ylabel("channels needed c")
ax.set_title("Minimum trunk size for a given grade of service"); ax.grid(alpha=0.3); ax.legend(); plt.show()
print("Channels for 2% blocking:", {a: tr.channels_for_gos(a, 0.02) for a in [1, 5, 10, 25, 50, 100]})

## 3. Trunking efficiency: splitting a trunk
Trunking works because users rarely all talk at once; it works *better* the larger the pool. Sectoring a cell with
$c$ channels into $S$ sectors gives $S$ separate trunks of $c/S$ channels each (a call cannot borrow a channel from the next
sector). The SIR improves (Week 2), but the carried traffic at a fixed GoS drops:
$$A_{\mathrm{cell}} = S\cdot A\!\left(\tfrac{c}{S}, \text{GoS}\right) < A(c,\text{GoS}), \qquad \eta_T = A/c .$$

In [ ]:
def show_sector_split(c_cell=57, gos=0.02):
    cs = np.arange(1, 201)
    eta = np.array([tr.offered_traffic_for_gos(k, gos) / k for k in cs])
    configs = [(1, "omni"), (3, "120° sectors"), (6, "60° sectors")]
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].plot(cs, eta, color="0.4", lw=1.5)
    totals = []
    for i, (S, name) in enumerate(configs):
        k = c_cell // S
        a = S * tr.offered_traffic_for_gos(k, gos) if k else 0.0
        totals.append(a)
        if k:
            axes[0].plot(k, eta[k - 1], "o", color=f"C{i}", ms=9, mec="white", mew=2, label=f"{name}: {k} ch/trunk")
    axes[0].set_xlabel("channels in the trunk c"); axes[0].set_ylabel(r"$\eta_T = A/c$ (Erlangs/channel)")
    axes[0].set_title(f"Trunking efficiency at GoS = {100*gos:g}%"); axes[0].grid(alpha=0.3); axes[0].legend(loc="lower right")
    bars = axes[1].bar([n for _, n in configs], totals, color=["C0", "C1", "C2"], edgecolor="white", linewidth=2)
    for b, a in zip(bars, totals):
        axes[1].text(b.get_x() + b.get_width() / 2, a, f"{a:.1f} Erl\n({100*a/totals[0]:.0f}%)", ha="center", va="bottom")
    axes[1].set_ylim(0, totals[0] * 1.25); axes[1].set_ylabel("traffic carried per cell (Erlangs)")
    axes[1].set_title(f"Same {c_cell} channels, split into sector trunks")
    plt.tight_layout(); plt.show()

interact(show_sector_split, c_cell=IntSlider(value=57, min=6, max=200, step=3, description="c per cell"),
         gos=FloatSlider(value=0.02, min=0.005, max=0.1, step=0.005, description="GoS"));

## 4. Discrete-time Markov chains
A Markov chain on $\mathcal{X} = \{0,1\}$ with transition matrix and initial distribution (from the slides)
$$\mathbf{P} = \begin{bmatrix} a & 1-a\\ b & 1-b\end{bmatrix}, \qquad \boldsymbol{\pi}_0 = \begin{bmatrix}\tfrac12 & \tfrac12\end{bmatrix}$$
evolves as $\boldsymbol{\pi}_{n+1} = \boldsymbol{\pi}_n\mathbf{P}$, i.e. $\boldsymbol{\pi}_n = \boldsymbol{\pi}_0\mathbf{P}^n$. Solving
$\boldsymbol{\pi} = \boldsymbol{\pi}\mathbf{P}$ with $\pi(0)+\pi(1)=1$ gives the stationary distribution
$$\boldsymbol{\pi} = \frac{1}{1-a+b}\begin{bmatrix} b & 1-a\end{bmatrix}.$$
The distance to $\boldsymbol{\pi}$ shrinks like $|a-b|^n$ ($a-b$ is the second eigenvalue of $\mathbf{P}$).
Left: $\boldsymbol{\pi}_n$ from the matrix recursion vs. the fraction of 2000 simulated chains in each state.
Right: the running time-average of **one** long sample path also converges to $\boldsymbol{\pi}$.

In [ ]:
def show_dtmc(a=0.9, b=0.3, pi0_0=0.5, n_steps=30):
    P = tr.two_state_P(a, b)
    pi = tr.two_state_stationary(a, b)
    ev = tr.dtmc_evolve([pi0_0, 1 - pi0_0], P, n_steps)
    # ensemble of independent chains, all started from pi_0
    M = 2000
    x = (rng.random(M) > pi0_0).astype(int)
    frac0 = [np.mean(x == 0)]
    for _ in range(n_steps):
        x = (rng.random(M) < P[x, 1]).astype(int)       # P[x,1] = Pr(next state = 1 | x)
        frac0.append(np.mean(x == 0))
    n = np.arange(n_steps + 1)
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
    ax = axes[0]
    ax.plot(n, ev[:, 0], "-", color="C0", lw=2, label=r"$\pi_n(0)$ (theory)")
    ax.plot(n, ev[:, 1], "-", color="C1", lw=2, label=r"$\pi_n(1)$ (theory)")
    ax.plot(n, frac0, "o", color="C0", ms=4, alpha=0.7, label=f"fraction in 0 ({M} chains)")
    ax.axhline(pi[0], color="C0", ls=":", lw=1); ax.axhline(pi[1], color="C1", ls=":", lw=1)
    ax.set_ylim(-0.02, 1.02); ax.set_xlabel("step n"); ax.set_ylabel("probability")
    ax.set_title(f"Evolution $\\pi_n = \\pi_0 P^n$  (dotted: $\\pi$ = [{pi[0]:.3f}, {pi[1]:.3f}])")
    ax.grid(alpha=0.3); ax.legend(fontsize=8)
    path = tr.simulate_dtmc(P, 5000, x0=0, rng=rng)
    k = np.arange(1, path.size + 1)
    axes[1].semilogx(k, np.cumsum(path == 0) / k, color="C0", lw=1.5, label="time-average fraction in 0")
    axes[1].axhline(pi[0], color="k", ls="--", lw=1, label=r"$\pi(0)$")
    axes[1].set_ylim(-0.02, 1.02); axes[1].set_xlabel("number of steps"); axes[1].set_ylabel("fraction of time in state 0")
    axes[1].set_title("One sample path (started in state 0)"); axes[1].grid(alpha=0.3, which="both"); axes[1].legend()
    plt.tight_layout(); plt.show()
    print("pi P =", np.round(pi @ P, 4), " pi =", np.round(pi, 4), " second eigenvalue a-b =", round(a - b, 3))

interact(show_dtmc, a=FloatSlider(value=0.9, min=0, max=1, step=0.05), b=FloatSlider(value=0.3, min=0.0, max=1, step=0.05),
         pi0_0=FloatSlider(value=0.5, min=0, max=1, step=0.05, description="π₀(0)"),
         n_steps=IntSlider(value=30, min=5, max=100, step=5));

Try $a = 0$, $b = 1$ (the chain alternates deterministically: eigenvalue $a-b=-1$, $\boldsymbol{\pi}_n$ oscillates forever unless
$\boldsymbol{\pi}_0 = \boldsymbol{\pi}$), and $a = 1$, $b = 0$ (two absorbing states: no unique stationary distribution).

## 5. The Poisson process
A Poisson process $\{A(t)\}$ with rate $\lambda$ has the properties used in the derivation:

* **P2** counts in disjoint intervals are independent;
* **P3** $\Pr\left(A(t,t+\tau]=n\right) = \dfrac{(\lambda\tau)^n}{n!}e^{-\lambda\tau}$;
* **P4** inter-arrival times $T_n$ are iid exponential: $f_{T_n}(s) = \lambda e^{-\lambda s}$, $\mathrm{E}[T_n] = 1/\lambda$, $\mathrm{Var}[T_n]=1/\lambda^2$;
* **P5** superposition of independent Poisson($\lambda_1$) and Poisson($\lambda_2$) is Poisson($\lambda_1+\lambda_2$);
* **P6** in a short interval $\epsilon$: $\Pr(\text{1 event}) \approx \lambda\epsilon$, $\Pr(\text{>1 event}) = o(\epsilon)$.

A bonus property (the dual of P5): **splitting** — keeping each event independently with probability $p$ gives a Poisson($p\lambda$) process.
We generate the process from exponential gaps (P4) and check everything else by simulation.

In [ ]:
def show_poisson(lam=2.0, tau=3.0, lam2=1.0, p_keep=0.3):
    T = 4000.0
    t1 = tr.poisson_process(lam, T, rng=rng)
    t2 = tr.poisson_process(lam2, T, rng=rng)
    fig, axes = plt.subplots(2, 3, figsize=(13, 7))
    # sample path A(t)
    ax = axes[0, 0]; tt = t1[t1 <= 10]
    ax.step(np.concatenate([[0], tt, [10]]), np.concatenate([[0], np.arange(1, tt.size + 1), [tt.size]]), where="post", color="C0")
    ax.plot(tt, np.zeros_like(tt), "*", color="C1", ms=10, clip_on=False)
    ax.set_xlim(0, 10); ax.set_xlabel("t (s)"); ax.set_ylabel("A(t)"); ax.set_title(f"Sample path, λ = {lam:g} /s")
    # P3 counts
    ax = axes[0, 1]; n = tr.counts_in_windows(t1, tau, T); k = np.arange(n.max() + 2)
    ax.bar(k, np.bincount(n, minlength=k.size) / n.size, color="C0", alpha=0.6, label="simulated")
    pmf = np.exp(k * np.log(lam * tau) - lam * tau - np.array([math.lgamma(i + 1) for i in k]))
    ax.plot(k, pmf, "o", color="C1", ms=6, label="Poisson(λτ)")
    ax.set_xlabel(f"events in a window of τ = {tau:g} s"); ax.set_ylabel("probability"); ax.set_title("P3: counts are Poisson"); ax.legend()
    # P4 inter-arrivals
    ax = axes[0, 2]; g = np.diff(t1); s = np.linspace(0, g.max(), 200)
    ax.hist(g, bins=60, density=True, color="C0", alpha=0.6, label="simulated")
    ax.plot(s, lam * np.exp(-lam * s), color="C1", lw=2, label=r"$\lambda e^{-\lambda s}$")
    ax.set_xlim(0, 5 / lam); ax.set_xlabel("inter-arrival time s (s)"); ax.set_ylabel("pdf"); ax.set_title("P4: exponential gaps"); ax.legend()
    # P2 independence
    ax = axes[1, 0]
    ax.plot(n[:-1] + rng.uniform(-0.3, 0.3, n.size - 1), n[1:] + rng.uniform(-0.3, 0.3, n.size - 1), ".", ms=3, alpha=0.4, color="C0")
    r = np.corrcoef(n[:-1], n[1:])[0, 1]
    ax.set_xlabel("count in window k"); ax.set_ylabel("count in window k+1"); ax.set_title(f"P2: disjoint windows, corr = {r:+.3f}")
    # P5 superposition
    ax = axes[1, 1]; gm = np.diff(np.sort(np.concatenate([t1, t2]))); lm = lam + lam2; s = np.linspace(0, 5 / lm, 200)
    ax.hist(gm, bins=60, density=True, color="C2", alpha=0.6, label="merged gaps")
    ax.plot(s, lm * np.exp(-lm * s), color="C1", lw=2, label=f"exp(λ₁+λ₂ = {lm:g})")
    ax.set_xlim(0, 5 / lm); ax.set_xlabel("inter-arrival time (s)"); ax.set_ylabel("pdf"); ax.set_title("P5: superposition"); ax.legend()
    # splitting
    ax = axes[1, 2]; kept = t1[rng.random(t1.size) < p_keep]; nk = tr.counts_in_windows(kept, tau, T); k = np.arange(nk.max() + 2)
    ax.bar(k, np.bincount(nk, minlength=k.size) / nk.size, color="C4", alpha=0.6, label="kept events")
    pmf = np.exp(k * np.log(p_keep * lam * tau) - p_keep * lam * tau - np.array([math.lgamma(i + 1) for i in k]))
    ax.plot(k, pmf, "o", color="C1", ms=6, label="Poisson(pλτ)")
    ax.set_xlabel(f"kept events per τ = {tau:g} s"); ax.set_ylabel("probability"); ax.set_title(f"Splitting, p = {p_keep:g}"); ax.legend()
    for a in axes.flat: a.grid(alpha=0.3)
    plt.tight_layout(); plt.show()
    print(f"mean gap = {g.mean():.4f} (1/λ = {1/lam:.4f}),  var = {g.var():.4f} (1/λ² = {1/lam**2:.4f})")

interact(show_poisson, lam=FloatSlider(value=2.0, min=0.2, max=5, step=0.1, description="λ (1/s)"),
         tau=FloatSlider(value=3.0, min=0.5, max=10, step=0.5, description="τ (s)"),
         lam2=FloatSlider(value=1.0, min=0.2, max=5, step=0.1, description="λ₂ (1/s)"),
         p_keep=FloatSlider(value=0.3, min=0.05, max=1, step=0.05, description="p (split)"));

**Property 6** is why the Markov chains have only nearest-neighbor transitions: in a window of length $\epsilon$,
two or more events happen with probability $o(\epsilon)$ — it vanishes faster than $\epsilon$ itself.

In [ ]:
lam = 2.0
eps = np.logspace(-3, 0, 40)
p1 = lam * eps * np.exp(-lam * eps)
p2 = 1 - np.exp(-lam * eps) - p1
fig, ax = plt.subplots(figsize=(7, 4))
ax.loglog(eps, p1, color="C0", lw=2, label=r"$\Pr(A(t,t+\epsilon]=1)$")
ax.loglog(eps, lam * eps, color="C0", ls=":", lw=1.5, label=r"$\lambda\epsilon$")
ax.loglog(eps, p2, color="C1", lw=2, label=r"$\Pr(A(t,t+\epsilon]>1)$  (slope 2: $o(\epsilon)$)")
ax.set_xlabel(r"window length $\epsilon$ (s)"); ax.set_ylabel("probability")
ax.set_title(f"Property 6 (λ = {lam:g} /s): simultaneous events are negligible")
ax.grid(True, which="both", alpha=0.3); ax.legend(); plt.show()

## 6. A one-channel trunk and the balance equations
One POTS line shared by voice and modem calls: $X(t)\in\{0,1\}$ channels busy. Calls arrive at rate $\lambda$, and a call in
progress ends at rate $\mu$; a call that arrives while the line is busy is **blocked**. In a short step $\epsilon$ the chain is a
DTMC with $P(0,1)=\lambda\epsilon$, $P(1,0)=\mu\epsilon$. Detailed balance, $\pi(0)\lambda = \pi(1)\mu$, together with
$\pi(0)+\pi(1)=1$ gives
$$\pi(0) = \frac{\mu}{\lambda+\mu},\qquad \pi(1) = \frac{\lambda}{\lambda+\mu} = P_{\mathrm{BLOCK}} .$$
The simulation shows both readings of $\pi(1)$: the **fraction of time** the line is busy, and the **fraction of arriving
calls** that are blocked (equal because Poisson arrivals "see time averages").

In [ ]:
def show_pots(lam_per_hr=4.0, hold_min=10.0):
    lam, mu = lam_per_hr / 60, 1 / hold_min                     # per minute
    sim = tr.simulate_mmc(lam, mu, 1, 20000, loss=True, rng=rng)
    pi1 = lam / (lam + mu)
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.6), gridspec_kw=dict(width_ratios=[2.2, 1]))
    ax = axes[0]; T = 8 * 60                                    # first 8 hours
    ok = ~sim.blocked
    st, en = sim.arrivals[ok], sim.arrivals[ok] + sim.service[ok]
    for s, e in zip(st[st < T], en[st < T]):
        ax.fill_between([s, min(e, T)], 0, 1, step="post", color="C0", alpha=0.5, lw=0)
    bl = sim.arrivals[sim.blocked & (sim.arrivals < T)]
    acc = sim.arrivals[ok & (sim.arrivals < T)]
    ax.plot(acc, np.full(acc.size, 1.1), "v", color="C2", ms=6, label="call accepted")
    ax.plot(bl, np.full(bl.size, 1.1), "x", color="C3", ms=7, mew=2, label="call blocked")
    ax.set_xlim(0, T); ax.set_ylim(0, 1.45); ax.set_yticks([0, 1]); ax.set_xlabel("time (min)"); ax.set_ylabel("X(t)")
    ax.set_title("Occupancy of a single line (shaded = busy)"); ax.legend(loc="upper center", ncol=2, fontsize=8, framealpha=1)
    ax = axes[1]
    labels = ["time busy", "calls blocked"]
    ax.bar(labels, [sim.state_frac[1], sim.blocking], color=["C0", "C3"], alpha=0.7, edgecolor="white", linewidth=2)
    ax.axhline(pi1, color="k", ls="--", lw=1.2, label=r"$\pi(1)=\lambda/(\lambda+\mu)$")
    ax.set_ylim(0, 1); ax.set_ylabel("fraction"); ax.set_title(f"Theory π(1) = {pi1:.3f}"); ax.legend(fontsize=8)
    plt.tight_layout(); plt.show()
    print(f"rho = {lam/mu:.3f} Erl;  simulated busy fraction = {sim.state_frac[1]:.3f}, blocked fraction = {sim.blocking:.3f}, "
          f"Erlang B(rho,1) = {tr.erlang_b(lam/mu, 1):.3f}")

interact(show_pots, lam_per_hr=FloatSlider(value=4.0, min=0.5, max=20, step=0.5, description="calls/hour"),
         hold_min=FloatSlider(value=10.0, min=1, max=60, step=1, description="1/μ (min)"));

**Global vs. detailed balance.** Global balance, $\pi(i)\sum_{j\ne i}P(i,j) = \sum_{j\ne i}\pi(j)P(j,i)$, holds for any chain; for a
birth–death chain it collapses to the much simpler detailed balance $\pi(i)P(i,i+1) = \pi(i+1)P(i+1,i)$. Below we solve both
for the $\epsilon$-discretized POTS chain (the answer does not depend on $\epsilon$) and for a bigger birth–death chain.

In [ ]:
lam, mu = 4 / 60, 1 / 10
for eps in [1.0, 0.1, 0.01]:
    P = np.array([[1 - lam * eps, lam * eps], [mu * eps, 1 - mu * eps]])
    print(f"eps = {eps:<5}: pi = pi P  ->  {np.round(tr.stationary_distribution(P), 4)}")
print(f"closed form           ->  {np.round([mu / (lam + mu), lam / (lam + mu)], 4)}")

rho, c = 6.0, 10
Q = tr.mmc_generator(rho, 1.0, c, c + 1)                 # M/M/10/10 generator (global balance: pi Q = 0)
pi_global = tr.generator_stationary(Q)
pi_detail = tr.birth_death_stationary(np.full(c, rho), np.arange(1, c + 1))
print(f"\nM/M/{c}/{c}, rho = {rho}: max |global - detailed| = {np.abs(pi_global - pi_detail).max():.2e}")

**Queuing notation** A/B/S/N: arrival process / service-time distribution / servers / waiting room (M = memoryless).
Erlang B is M/M/$c$/$c$ FCFS; Erlang C is M/M/$c$/$\infty$ FCFS. Both are birth–death chains with birth rate $\lambda$ and death rate
$\min(i,c)\,\mu$ (P5: $i$ calls in progress finish at total rate $i\mu$).

## 7. Erlang B: the M/M/$c$/$c$ loss system
States $0,\dots,c$ (channels busy), births $\lambda$, deaths $i\mu$. Detailed balance $\lambda\pi(i-1) = i\mu\,\pi(i)$ gives
$$\pi(i) = \frac{\rho^i}{i!}\pi(0),\qquad \pi(0) = \left[\sum_{i=0}^{c}\frac{\rho^i}{i!}\right]^{-1},\qquad B(\rho,c) = \pi(c).$$
The simulation below is an event-driven M/M/$c$/$c$ with blocked calls cleared (holding time $1/\mu = 3$ min).

In [ ]:
HOLD = 3.0                                    # mean holding time (min)

def show_mmcc(rho=8.0, c=10, n_calls=40000):
    sim = tr.simulate_mmc(rho / HOLD, 1 / HOLD, c, n_calls, loss=True, rng=rng)
    pi = tr.mmcc_stationary(rho, c)
    i = np.arange(c + 1)
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.bar(i, sim.state_frac, color="C0", alpha=0.6, edgecolor="white", label="simulated fraction of time")
    ax.plot(i, pi, "o", color="C1", ms=7, label=r"$\pi(i)=\frac{\rho^i}{i!}\pi(0)$")
    ax.bar([c], [sim.state_frac[c]], color="C3", alpha=0.7, edgecolor="white", label="state c: all channels busy")
    ax.set_xlabel("channels busy i"); ax.set_ylabel("probability"); ax.grid(axis="y", alpha=0.3)
    ax.set_title(f"M/M/{c}/{c}, ρ = {rho:g} Erl: B(ρ,c) = {tr.erlang_b(rho, c):.4f} (theory), "
                 f"{sim.blocking:.4f} of {n_calls} calls blocked (sim)")
    ax.legend(); plt.show()
    carried = rho * (1 - tr.erlang_b(rho, c))
    print(f"carried traffic = rho (1 - B) = {carried:.3f} Erl = mean channels busy (sim: {np.dot(i, sim.state_frac):.3f})")

interact(show_mmcc, rho=FloatSlider(value=8.0, min=0.5, max=30, step=0.5, description="ρ (Erl)"),
         c=IntSlider(value=10, min=1, max=30), n_calls=IntSlider(value=40000, min=5000, max=200000, step=5000));

In [ ]:
# Theory vs simulation across load, with 95% confidence intervals from 5 independent runs per point
rhos = np.linspace(1, 16, 11)
fig, ax = plt.subplots(figsize=(8, 4.5))
for k, c in enumerate([5, 10, 15]):
    r_fine = np.linspace(0.5, 16, 200)
    ax.semilogy(r_fine, tr.erlang_b(r_fine, c), color=f"C{k}", lw=2, label=f"Erlang B, c = {c}")
    runs = np.array([[tr.simulate_mmc(r / HOLD, 1 / HOLD, c, 20000, loss=True, rng=rng).blocking for _ in range(5)] for r in rhos])
    m, ci = runs.mean(1), 1.96 * runs.std(1, ddof=1) / np.sqrt(5)
    ax.errorbar(rhos, m, yerr=ci, fmt="o", color=f"C{k}", ms=5, capsize=3, mec="white")
ax.set_ylim(1e-4, 1); ax.set_xlabel("offered traffic ρ (Erlangs)"); ax.set_ylabel("blocking probability")
ax.set_title("Erlang B: formula (lines) vs discrete-event simulation (dots, 95% CI)")
ax.grid(True, which="both", alpha=0.3); ax.legend(loc="lower right"); plt.show()

## 8. Erlang C: the M/M/$c$/$\infty$ delay system
Now blocked calls wait in an infinite FCFS queue. For $i\le c$, $\pi(i) = \frac{\rho^i}{i!}\pi(0)$; beyond $c$ only $c$ calls are
served, so $\pi(i) = \frac{c^c}{c!}\left(\frac{\rho}{c}\right)^i\pi(0)$ (a geometric tail; needs $\rho < c$). Then
$$C(\rho,c) = \sum_{i=c}^{\infty}\pi(i),\qquad \Pr(\text{Wait}>t \mid \text{delayed}) = e^{(\lambda-c\mu)t},$$
$$N_Q = \sum_{i=c}^{\infty}(i-c)\pi(i) = C(\rho,c)\frac{\rho}{c-\rho},\qquad \text{Little: } W_Q = N_Q/\lambda = \frac{C(\rho,c)}{c\mu-\lambda}.$$

In [ ]:
def show_mmc(rho=8.0, c=10, n_calls=150000):
    if rho >= c:
        print("Unstable: need rho < c (the queue grows without bound)."); return
    lam, mu = rho / HOLD, 1 / HOLD
    sim = tr.simulate_mmc(lam, mu, c, n_calls, loss=False, rng=rng)
    nmax = int(c + 6 * rho / (c - rho) + 10)
    pi = tr.mmc_stationary(rho, c, nmax)
    i = np.arange(nmax + 1)
    sf = np.zeros(nmax + 1); m = min(sf.size, sim.state_frac.size); sf[:m] = sim.state_frac[:m]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    ax = axes[0]
    ax.bar(i, sf, color=np.where(i < c, "C0", "C3"), alpha=0.6, edgecolor="white", linewidth=0.5)
    ax.plot(i, pi, "o", color="C1", ms=4, label="theory π(i)")
    ax.axvline(c - 0.5, color="k", lw=1, ls=":")
    ax.text(c - 0.3, ax.get_ylim()[1] * 0.92, "queueing (red) →", fontsize=9)
    ax.set_xlabel("calls in system i"); ax.set_ylabel("probability"); ax.legend()
    ax.set_title(f"M/M/{c}, ρ = {rho:g}: C = {tr.erlang_c(rho, c):.4f} (theory), {sim.delayed:.4f} delayed (sim)")
    ax = axes[1]
    t = np.linspace(0, max(4 / (c * mu - lam), 1), 200)
    w = np.sort(sim.wait)
    ax.semilogy(t, np.mean(w[None, :] > t[:, None], axis=1), color="C0", lw=2, label="simulated Pr(W > t)")
    ax.semilogy(t, tr.erlang_c_wait_tail(t, lam, mu, c), "--", color="C1", lw=2, label=r"$C(\rho,c)\,e^{-(c\mu-\lambda)t}$")
    ax.set_xlabel("waiting time t (min)"); ax.set_ylabel("Pr(W > t)"); ax.set_title("Waiting-time tail (all calls)")
    ax.grid(True, which="both", alpha=0.3); ax.legend()
    plt.tight_layout(); plt.show()
    print(f"N_Q: theory {tr.erlang_c_mean_queue(rho, c):.3f}, sim {np.dot(np.clip(i - c, 0, None), sf):.3f}   |   "
          f"W_Q: theory {tr.erlang_c_mean_wait(lam, mu, c):.3f} min, sim {np.mean(sim.wait):.3f} min")

interact(show_mmc, rho=FloatSlider(value=8.0, min=0.5, max=19.5, step=0.5, description="ρ (Erl)"),
         c=IntSlider(value=10, min=1, max=20), n_calls=IntSlider(value=150000, min=5000, max=200000, step=5000));

**Same trunk, two policies.** For $c=10$ the probability that a call cannot be served immediately is always higher when calls
wait (Erlang C) than when they are cleared (Erlang B), and $C\to1$ as $\rho\to c$ while $B$ stays moderate.

In [ ]:
c = 10
r = np.linspace(0.5, 9.95, 300)
fig, ax = plt.subplots(figsize=(8, 4.2))
ax.plot(r, tr.erlang_b(r, c), color="C0", lw=2, label="Erlang B: Pr(blocked)")
ax.plot(r, tr.erlang_c(r, c), color="C3", lw=2, label="Erlang C: Pr(delayed)")
ax.set_xlabel("offered traffic ρ (Erlangs)"); ax.set_ylabel("probability"); ax.set_ylim(0, 1)
ax.set_title(f"c = {c} channels: cleared vs delayed"); ax.grid(alpha=0.3); ax.legend(); plt.show()

## Try it yourself
* In Section 1, lower the SIR requirement from 18 dB to 12 dB (e.g. a better modem with a narrower $W_{\mathrm{ch}}$). Which cluster
  size becomes optimal, and by how much does $\eta_S$ improve? Which of $\eta_B$, $\eta_C$, $\eta_T$ changed?
* A cell has 57 channels and 120° sectoring. Using Section 3, how many *extra* channels per cell would be needed to recover the
  omni cell's carried traffic at 2% GoS? Is sectoring still worth it once the SIR gain lets you drop from $N=7$ to $N=4$?
* In Section 4, what does $|a-b|$ tell you about how many steps the chain needs to "forget" $\boldsymbol{\pi}_0$? Find $(a,b)$ with the
  same $\boldsymbol{\pi}$ but very different convergence speeds.
* The Erlang formulas assume Poisson arrivals from an infinite population. Modify `tr.simulate_mmc` (or write your own) so
  that only $U = 20$ users generate calls and a busy user cannot call again (the Engset model). How does blocking compare
  with $B(\rho,c)$ at the same offered traffic?